In [22]:
import json
from collections import OrderedDict
original_fp="/home/jovyan/pscratch/suffix-tree-decoding/trace/llama70b/cortex-llama3.1-70b.json"
new_fp="/home/jovyan/pscratch/suffix-tree-decoding/trace/llama70b/cortex-llama3.1-70b_debug.json"

with open(original_fp, "r") as f:
    original_data = json.load(f, object_pairs_hook=OrderedDict)
partitions = original_data["partitions"]
# remove partitions whose partition_name does not match "FEATURE_EXTRACTION"
partitions = [p for p in partitions if p["partition_name"] == "FEATURE_EXTRACTION"]
assert len(partitions) == 1
partitions[0]["eval_entries"] = [e for e in partitions[0]["eval_entries"] if e["prompt_length"] == 967]
partitions[0]["eval_entries"] = partitions[0]["eval_entries"][:1]

new_json = OrderedDict({
    "partitions": partitions,
    "metadata": original_data["metadata"],
})

# save data back to json
with open(new_fp, "w") as f:
    json.dump(new_json, f, indent=2)

In [23]:
import os
os.environ["HF_HOME"] = "/home/jovyan/pscratch/.cache/huggingface"
# tokenize a sentence with llama 3.1 70b
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.1-70B-Instruct")
sentence = " \n  \"ratio_question\": false, \n  \"timed\": true,\n  \"rolling_metric_question\": false,\n  \"rank_calculation\": false\n}\n```"
correct_output=" \n  \"ratio_question\": true, \n  \"last_period\": false, \n  \"timedimension_aggregation\": true, \n  \"missing_time_period\": false, \n  \"period_over_period_question\": true, \n  \"rolling_metric_question\": false, \n  \"consecutive_trend_question\": false, \n  \"rank_calculation\": false \n}\n```"
tokens = tokenizer.encode(sentence, add_special_tokens=False)
print(tokens)

[720, 220, 330, 46458, 30015, 794, 905, 11, 720, 220, 330, 20693, 291, 794, 837, 345, 220, 330, 16608, 42394, 30015, 794, 905, 345, 220, 330, 13430, 39341, 2987, 794, 905, 198, 534, 74694]


In [24]:
tokenizer.decode([720, 220, 330, 46458, 30015, 794, 905, 11, 720, 220, 330, 20693])

' \n  "ratio_question": false, \n  "tim'

In [26]:
tokens=tokenizer.encode(correct_output, add_special_tokens=False)
print(tokens)

[720, 220, 330, 46458, 30015, 794, 837, 11, 720, 220, 330, 4354, 21485, 794, 905, 11, 720, 220, 330, 20693, 291, 18658, 21233, 35542, 794, 837, 11, 720, 220, 330, 31716, 3084, 21485, 794, 905, 11, 720, 220, 330, 19862, 15793, 21485, 30015, 794, 837, 11, 720, 220, 330, 16608, 42394, 30015, 794, 905, 11, 720, 220, 330, 444, 86880, 530, 9484, 30015, 794, 905, 11, 720, 220, 330, 13430, 39341, 2987, 794, 905, 720, 534, 74694]
